In [6]:
import math
import random

import torch
import torchvision.models as models
import numpy as np

In [ ]:
model = models.mobilenet_v3_small(pretrained=True)
model.eval()
print(model)

/home/vitya/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/vitya/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /home/vitya/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100%|██████████| 9.83M/9.83M [00:00<00:00, 11.0MB/s]

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [10]:
model.classifier = torch.nn.Identity()

In [14]:
fake_image = torch.randn(1, 3, 480, 640)

with torch.no_grad():
    predict = model(fake_image)

In [12]:
predict.shape

torch.Size([1, 576])

In [18]:
fake_image = np.random.rand(480, 640, 3)
fake_image_tensor = torch.from_numpy(fake_image).permute(2, 0, 1).unsqueeze(0).float()
fake_image_tensor.shape, fake_image_tensor.dtype

(torch.Size([1, 3, 480, 640]), torch.float32)

In [21]:
with torch.no_grad():
    emb = model(fake_image_tensor).squeeze(0).numpy()
    print(emb.shape, emb.dtype)
    

(576,) float32


In [3]:
arm_pose = np.array([0.956514, 0.126503, 0.407556])
cube_pose = np.array([1.5, 0, 0])

dist = np.linalg.norm(cube_pose - arm_pose)
print(dist)

0.6910006760785404


In [8]:
for i in range(10):
        angle = math.radians(random.uniform(-45, 45))
        radius = 1.5
        x = radius * math.cos(angle)
        y = radius * math.sin(angle)
        z = 0.05
        print(x, y, z)
        print(np.linalg.norm(np.array([x, y, z])))

1.2923806360334844 0.7614146646904604 0.05
1.5008331019803633
1.488360379924451 -0.18650302804819024 0.05
1.5008331019803633
1.492674486524543 0.1480637608555604 0.05
1.5008331019803638
1.4925484892028993 0.14932852164989385 0.05
1.5008331019803633
1.3570022289963517 0.6391752111111111 0.05
1.5008331019803636
1.4420805104832766 0.41279995310597195 0.05
1.5008331019803633
1.3609459151599492 0.6307346637132355 0.05
1.5008331019803633
1.1407760400762352 0.973976399297223 0.05
1.5008331019803633
1.418180427971893 -0.4886351130623523 0.05
1.5008331019803633
1.4908107767568437 0.16578066203769393 0.05
1.5008331019803633


In [5]:
np.linalg.norm(np.array([0.970728, -0.565225, 0.049995]))

1.1244072441219863

In [ ]:



# angle = random.uniform(-90 / 2, 90 / 2)
angle = math.radians(45)
radius = 1.5
x = radius * math.cos(angle)
y = radius * math.sin(angle)
z = 0.05

x, y, z

(1.0606601717798214, 1.0606601717798212, 0.05)

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time
import os

def live_plot_training_progress(csv_file, update_interval=3):
    """
    Динамически обновляемые графики в Jupyter Notebook
    
    Параметры:
        csv_file (str): путь к CSV файлу с логами
        update_interval (int): интервал обновления в секундах
    """
    plt.figure(figsize=(12, 10))
    
    try:
        while True:
            # Проверяем существование файла
            if not os.path.exists(csv_file):
                print(f"Ожидание файла {csv_file}...")
                time.sleep(update_interval)
                continue
                
            # Читаем данные
            try:
                df = pd.read_csv(csv_file)
            except:
                time.sleep(update_interval)
                continue
                
            if df.empty:
                time.sleep(update_interval)
                continue
                
            # Очищаем вывод ячейки
            clear_output(wait=True)
            
            # Создаем новые графики
            plt.clf()
            
            # График 1: Награда за эпизод
            plt.subplot(3, 1, 1)
            plt.plot(df['episode'], df['episode_reward'], 'b-')
            plt.title(f'Награда за эпизод (последняя: {df["episode_reward"].iloc[-1]:.2f})')
            plt.ylabel('Награда')
            plt.grid(True)
            
            # График 2: Средняя награда
            plt.subplot(3, 1, 2)
            plt.plot(df['episode'], df['mean_reward_100'], 'r-')
            plt.title(f'Средняя награда (последние 100: {df["mean_reward_100"].iloc[-1]:.2f})')
            plt.ylabel('Награда')
            plt.grid(True)
            
            # График 3: Длина эпизода
            plt.subplot(3, 1, 3)
            plt.plot(df['episode'], df['episode_length'], 'g-', label='Длина')
            plt.plot(df['episode'], df['mean_length_100'], 'orange', label='Средняя длина (100)')
            plt.title('Длина эпизодов')
            plt.xlabel('Эпизод')
            plt.ylabel('Шаги')
            plt.legend()
            plt.grid(True)
            
            plt.tight_layout()
            plt.show()
            
            time.sleep(update_interval)
            
    except KeyboardInterrupt:
        print("Визуализация остановлена")

In [16]:
live_plot_training_progress('/home/vitya/diploma/roboarm/src/roboarm_rl/training_logs/20250515/training_log_20250515_030044.csv')

Визуализация остановлена
